# 104. Visual Question Answering: Q&A About Images

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amerob/ultimate-prompt-engineering-playbook/blob/main/notebooks/13-multi-modal/104_visual_question_answering.ipynb)

**Category:** 13 - Multi-Modal Techniques  
**Technique #:** 104  
**Difficulty:** Intermediate

## 📖 Description

Visual Question Answering (VQA) is a multi-modal AI technique that combines computer vision and natural language processing to answer questions about images. The model analyzes visual content and provides textual answers based on what it sees.

### When to Use:
- Answering specific questions about image content
- Interactive image exploration
- Accessibility applications for visually impaired users
- Educational tools and tutorials
- Automated image annotation and tagging

## 🔧 How It Works

```
┌─────────────────────────────────────────────────────────────┐
│              VISUAL QUESTION ANSWERING FLOW                  │
└─────────────────────────────────────────────────────────────┘

    ┌──────────────┐         ┌──────────────┐
    │    Image     │────────▶│   Visual     │
    │    Input     │         │   Encoder    │
    └──────────────┘         └──────┬───────┘
                                    │
                                    ▼
    ┌──────────────┐         ┌──────────────┐         ┌──────────────┐
    │   Question   │────────▶│  Cross-Modal │────────▶│   Answer     │
    │   (Text)     │         │  Reasoning   │         │   (Text)     │
    └──────────────┘         └──────────────┘         └──────────────┘

         Input                          Fusion                Output
```

### Process Flow:
1. **Image Encoding**: Visual features extracted from the image
2. **Question Processing**: Text question converted to embeddings
3. **Cross-Modal Fusion**: Visual and textual features combined
4. **Answer Generation**: Model generates relevant answer

## 🛠️ Setup

In [ ]:
# Install required packages
!pip install -q openai pillow requests

In [ ]:
import os
from getpass import getpass
import base64
import requests

# Get API key securely
api_key = getpass("Enter your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key

from openai import OpenAI
client = OpenAI()

## 💡 Basic Example

In [ ]:
def encode_image(image_path_or_url):
    """Encode image to base64."""
    if image_path_or_url.startswith(('http://', 'https://')):
        response = requests.get(image_path_or_url)
        return base64.b64encode(response.content).decode('utf-8')
    with open(image_path_or_url, "rb") as f:
        return base64.b64encode(f.read()).decode('utf-8')

def visual_qa(image_source, question, model="gpt-4o"):
    """Answer a question about an image."""
    base64_image = encode_image(image_source)
    
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": question},
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{base64_image}"
                            }
                        }
                    ]
                }
            ],
            max_tokens=500
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {str(e)}"

# Basic example
sample_image = "https://images.unsplash.com/photo-1565299624946-b28f40a0ae38?w=800"  # Pizza image

questions = [
    "What food is shown in this image?",
    "What colors are visible?",
    "Is this a healthy meal?"
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {visual_qa(sample_image, q)}")
    print("-" * 50)

## 🌍 Real-World Example

In [ ]:
# Real-world: E-commerce product Q&A
product_image = "https://images.unsplash.com/photo-1523275335684-37898b6baf30?w=800"  # Watch image

product_questions = [
    "What type of product is this?",
    "What color is the watch face?",
    "Does it have a leather strap?",
    "Is this suitable for formal occasions?",
    "What brand might this be based on the design?"
]

print("PRODUCT VISUAL Q&A SYSTEM")
print("="*60 + "\n")

for q in product_questions:
    answer = visual_qa(product_image, q)
    print(f"❓ {q}")
    print(f"✅ {answer}\n")

## ❌ Failure Case

In [ ]:
# Failure case: Questions requiring information not in the image
failure_image = "https://images.unsplash.com/photo-1542291026-7eec264c27ff?w=800"  # Red sneaker

problematic_questions = [
    "What is the price of these shoes?",  # Not in image
    "When was this photo taken?",  # No timestamp
    "Who is wearing these shoes?",  # No person visible
    "What store sells these?"  # Not identifiable from image
]

print("QUESTIONS THAT REQUIRE EXTERNAL KNOWLEDGE:")
print("="*60 + "\n")

for q in problematic_questions:
    answer = visual_qa(failure_image, q)
    print(f"❓ {q}")
    print(f"⚠️  {answer}\n")

print("\n" + "="*60)
print("LESSON: VQA can only answer based on VISUAL information.")
print("Questions requiring external knowledge need additional context.")

## 📊 Benchmark Comparison

| Dataset | GPT-4o | Claude 3.5 | Gemini 1.5 | Human |
|---------|--------|------------|------------|-------|
| VQAv2 | 77.2% | 75.8% | 76.5% | 80.8% |
| GQA | 63.8% | 62.1% | 64.2% | 89.3% |
| TextVQA | 78.5% | 76.2% | 77.8% | 85.0% |
| OK-VQA | 58.6% | 56.4% | 57.9% | - |

### Question Type Performance:
- **Object Recognition**: 90%+ accuracy
- **Color/Attribute**: 85%+ accuracy
- **Counting**: 75-80% accuracy
- **Spatial Relations**: 70-75% accuracy
- **Reasoning**: 60-65% accuracy

## 🎮 Interactive Playground

In [ ]:
def vqa_playground():
    """Interactive VQA playground."""
    print("\n" + "="*60)
    print("VISUAL QUESTION ANSWERING PLAYGROUND")
    print("="*60 + "\n")
    
    # Image selection
    print("Choose an image:")
    print("1. Nature landscape")
    print("2. City scene")
    print("3. Food")
    print("4. Custom URL")
    
    choice = input("Enter choice (1-4): ").strip()
    
    images = {
        "1": "https://images.unsplash.com/photo-1506905925346-21bda4d32df4?w=800",
        "2": "https://images.unsplash.com/photo-1449824913935-59a10b8d2000?w=800",
        "3": "https://images.unsplash.com/photo-1504674900247-0877df9cc836?w=800",
        "4": None
    }
    
    image_url = images.get(choice)
    if choice == "4" or image_url is None:
        image_url = input("Enter image URL: ").strip()
    
    print("\nAsk questions about the image (type 'quit' to exit):\n")
    
    while True:
        question = input("Your question: ").strip()
        if question.lower() == 'quit':
            break
        if question:
            answer = visual_qa(image_url, question)
            print(f"🤖 {answer}\n")

# Run playground
vqa_playground()

## 💡 Tips & Tricks

### Question Formulation:
- **Be specific**: "What color is the car?" > "Describe the car"
- **Use clear language**: Avoid ambiguous terms
- **One question at a time**: Break complex queries into parts

### Handling Uncertainty:
- Ask model to indicate confidence
- Request "I don't know" for unanswerable questions
- Use follow-up questions for clarification

### Performance Optimization:
- Use high-quality images (min 512x512)
- Crop to relevant regions when possible
- Pre-process images for better visibility

## 📚 References

1. [VQA Dataset](https://visualqa.org/)
2. [GQA Dataset](https://cs.stanford.edu/people/dorarad/gqa/)
3. [OpenAI Vision API](https://platform.openai.com/docs/guides/vision)
4. [Visual Question Answering: A Survey](https://arxiv.org/abs/2206.02087)